# Text Classification

1) Setup path  
2) (Proof) Load dataset via `datasets.load_dataset`  
3) Load via `modules.data_loader.load_data` (single source of truth for pipeline)  
4) TF‑IDF (subset) → Save features  
5) Train/Eval Logistic Regression baseline → Pretty output  


In [1]:
# 0) Quick environment check (optional but useful when debugging)
import os
from pathlib import Path

print("CWD:", Path.cwd())
print("Parent:", Path.cwd().parent)
print("Root list:", os.listdir(Path.cwd().parent) if Path.cwd().name == "notebooks" else os.listdir(Path.cwd()))


CWD: c:\Users\Admin\Documents\GitHub\MachineLearning_TextModule\notebooks
Parent: c:\Users\Admin\Documents\GitHub\MachineLearning_TextModule
Root list: ['.git', '.gitignore', '.venv', 'features', 'LICENSE', 'modules', 'notebooks', 'README.md', 'report.tex']


In [2]:
# 1) One-time setup: add project root to sys.path so imports from ../modules work
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebook is in notebooks/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: c:\Users\Admin\Documents\GitHub\MachineLearning_TextModule


## 2) Dependencies (install only if missing)



In [3]:
def _ensure_import(pkg: str) -> bool:
    try:
        __import__(pkg)
        return True
    except ImportError:
        return False

missing = []
if not _ensure_import("datasets"):
    missing.append("datasets")
if not _ensure_import("sklearn"):
    missing.append("scikit-learn")
if not _ensure_import("numpy"):
    missing.append("numpy")

if missing:
    import subprocess, sys as _sys
    print("Installing:", missing)
    subprocess.check_call([_sys.executable, "-m", "pip", "install", "-q"] + missing)
else:
    print("All required packages already installed.")


c:\Users\Admin\Documents\GitHub\MachineLearning_TextModule\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All required packages already installed.


## 3) Proof: load dataset directly from Hugging Face



In [4]:
from datasets import load_dataset

ds = load_dataset("ag_news")
print(ds)
print(ds["train"][0])


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


## 4) Pipeline load (via module)



In [5]:
from modules.data_loader import load_data

train_texts, train_labels, test_texts, test_labels, info = load_data("ag_news")
print(info)
print("Train:", len(train_texts), "Test:", len(test_texts))
print("Sample text:", train_texts[0][:120] + ("..." if len(train_texts[0]) > 120 else ""))
print("Sample label:", train_labels[0])


DatasetInfo(name='ag_news', train_size=120000, test_size=7600, text_field='text', label_field='label', num_classes=4)
Train: 120000 Test: 7600
Sample text: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics,...
Sample label: 2


## 5) TF‑IDF features 



In [6]:
from modules.tfidf_features import build_tfidf_features, save_features_npy

N_TRAIN = 5000
N_TEST = 2000

X_train, X_test, _ = build_tfidf_features(
    train_texts[:N_TRAIN],
    test_texts[:N_TEST],
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)

# Save into ROOT/features (not notebooks/features)
train_path, test_path = save_features_npy(
    X_train, X_test,
    feature_dir=str(PROJECT_ROOT / "features"),
    train_name="tfidf_train.npy",
    test_name="tfidf_test.npy",
)

import os
print("Saved:", train_path)
print("Saved:", test_path)
print("Files exist?", os.path.exists(train_path), os.path.exists(test_path))


X_train: (5000, 10000) X_test: (2000, 10000)
Saved: c:\Users\Admin\Documents\GitHub\MachineLearning_TextModule\features\tfidf_train.npy
Saved: c:\Users\Admin\Documents\GitHub\MachineLearning_TextModule\features\tfidf_test.npy
Files exist? True True


## 6) Train/Eval baseline (Logistic Regression)



In [7]:
from modules.train_classical import train_eval_logreg, pretty_print_result
import pandas as pd

y_train_sub = train_labels[:N_TRAIN]
y_test_sub = test_labels[:N_TEST]

result = train_eval_logreg(X_train, y_train_sub, X_test, y_test_sub)
metrics = pretty_print_result(result)

display(pd.DataFrame([metrics]))


c:\Users\Admin\Documents\GitHub\MachineLearning_TextModule\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


,accuracy,f1_weighted
0,0.8505,0.850286


8/3/2026


## 7) Embedding


In [8]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modules.bert_embed import EmbedConfig, get_or_build_embeddings

# Demo nhanh (week 1): chạy ít để không lâu
N_TRAIN_EMB = 2000
N_TEST_EMB = 500

emb_cfg = EmbedConfig(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    batch_size=32,
    normalize=True,
    device=None,  # auto
)

emb_train, emb_test, p_train, p_test = get_or_build_embeddings(
    train_texts[:N_TRAIN_EMB],
    test_texts[:N_TEST_EMB],
    feature_dir=str(PROJECT_ROOT / "features"),
    train_name="bert_train.npy",
    test_name="bert_test.npy",
    cfg=emb_cfg,
    rebuild=False
)

print("Embedding shapes:", emb_train.shape, emb_test.shape)
print("Saved:", p_train, p_test)
print("Exists:", os.path.exists(p_train), os.path.exists(p_test))

c:\Users\Admin\Documents\GitHub\MachineLearning_TextModule\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7718.11it

Embedding shapes: (2000, 384) (500, 384)
Saved: c:\Users\Admin\Documents\GitHub\MachineLearning_TextModule\features\bert_train.npy c:\Users\Admin\Documents\GitHub\MachineLearning_TextModule\features\bert_test.npy
Exists: True True


In [ ]:
from modules.train_classical import train_eval_logreg, pretty_print_result
import pandas as pd

# Labels khớp subset embedding
y_train_emb = train_labels[:N_TRAIN_EMB]
y_test_emb = test_labels[:N_TEST_EMB]

res_emb = train_eval_logreg(emb_train, y_train_emb, emb_test, y_test_emb)
row_emb = dict(pretty_print_result(res_emb))
row_emb["method"] = "SBERT embedding + LogReg"

# (Nếu bạn đã có TF-IDF baseline trước đó)
# Giả sử bạn chạy TF-IDF trên N_TRAIN, N_TEST và đã có result tfidf_result
# row_tfidf = dict(pretty_print_result(tfidf_result)); row_tfidf["method"]="TF-IDF + LogReg"

display(pd.DataFrame([row_emb]))  # hoặc DataFrame([row_tfidf, row_emb])